In [ ]:
# In[1]:

import pandas as pd

# Load CSVs
metric_df = pd.read_csv("metric.csv")
trace_df = pd.read_csv("trace.csv")
log_df = pd.read_csv("log.csv")
error_df = pd.read_csv("error_logs.csv")

# Parse timestamps as UTC datetimes (per rule 10)
metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)
trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)
log_df['timestamp'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True)
error_df['timestamp'] = pd.to_datetime(error_df['timestamp'], unit='s', utc=True)

# Ensure 'value' numeric where present
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')
log_df['value'] = pd.to_numeric(log_df['value'], errors='coerce')

# Helper to aggregate grouped KPI data
def aggregate_kpi(df, group_cols, value_col='value'):
    g = df.groupby(group_cols)[value_col]
    agg = g.agg(count='count',
                min='min',
                median=lambda x: x.quantile(0.5),
                mean='mean',
                max='max',
                p95=lambda x: x.quantile(0.95)).reset_index()
    # order columns for readability
    cols = group_cols + ['count', 'min', 'median', 'mean', 'max', 'p95']
    return agg[cols]

# Aggregations for each file (full file, no time filtering)
metric_agg = aggregate_kpi(metric_df, ['cmdb_id', 'kpi_name'])
trace_agg = aggregate_kpi(trace_df, ['cmdb_id', 'trace_name'])
log_agg = aggregate_kpi(log_df, ['cmdb_id', 'log_name'])

# Top 50 by count (descending) for each aggregated table
metric_agg_top = metric_agg.sort_values('count', ascending=False).head(50)
trace_agg_top = trace_agg.sort_values('count', ascending=False).head(50)
log_agg_top = log_agg.sort_values('count', ascending=False).head(50)

# Unique cmdb_id values across all files
unique_cmdbs = pd.Series(
    sorted(set(metric_df['cmdb_id'].dropna().unique())
           | set(trace_df['cmdb_id'].dropna().unique())
           | set(log_df['cmdb_id'].dropna().unique())
           | set(error_df['cmdb_id'].dropna().unique())),
    name='cmdb_id'
).to_frame()

# First 5 raw rows of each file for inspection
metric_head = metric_df.head(5)
trace_head = trace_df.head(5)
log_head = log_df.head(5)
error_head = error_df.head(5)

# Display the compact results (multiple variables separated by commas per instructions)
metric_agg_top, trace_agg_top, log_agg_top, unique_cmdbs, metric_head, trace_head, log_head, error_head

```
Out[1]:
```
summary = (
    "Summary of telemetry aggregation (full files, no time filtering):\n\n"
    "- Metric aggregations (top examples):\n"
    "  * ts-voucher-service / kpi 'socket': count=25, values constant at 4.\n"
    "  * ts-admin-basic-info-service / kpi 'diskio': count=25, min≈42279.93, median≈52826.28, "
    "mean≈52618.55, max≈63980.84, p95≈63454.34.\n"
    "  * Many metric groups have 25 samples (dataset appears to be 25 minutes long for most series).\n\n"
    "- Trace aggregations (top examples):\n"
    "  * root / trace.to_ts-admin-basic-info-service.row_count: count=25, min=3, median=11, mean=10.8, max=18, p95=16.8.\n"
    "  * Numerous trace groups (root and service-originated traces) also show count=25.\n\n"
    "- Log aggregations (top examples):\n"
    "  * ts-basic-service / log.row_count: count=25, mean≈2362.4 (high row throughput).\n"
    "  * log.error_count is 0 for many services (e.g., ts-admin-basic-info-service and others have error_count=0 with count=25).\n\n"
    "- Inventory and raw data preview:\n"
    "  * Unique cmdb_id values across all files: 69 distinct IDs (examples include 'root', many ts- services).\n"
    "  * First 5 rows were inspected for each file: metric/trace/log show expected timestamped rows; error_logs is empty (no rows).\n\n"
    "Notes: Aggregations were computed across the entire files (global thresholds not applied). The outputs show which service/KPI groups have the most samples and highlight a few notable KPI values (e.g., high diskio for ts-admin-basic-info-service and very high log.row_count for ts-basic-service)."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(                         cmdb_id    kpi_name  count           min        median          mean           max           p95
360           ts-voucher-service      socket     25      4.000000      4.000000      4.000000      4.000000      4.000000
0    ts-admin-basic-info-service         cpu     25      0.287109      0.391499      0.435889      0.950000      0.851091
1    ts-admin-basic-info-service      diskio     25  42279.926751  52826.277040  52618.552204  63980.836367  63454.335781
2    ts-admin-basic-info-service       error     25      0.000000      0.000000      0.000000      0.000000      0.000000
3    ts-admin-basic-info-service  latency-50     25      0.016080      0.017025      0.016972      0.017506      0.017473
..                           ...         ...    ...           ...           ...           ...           ...           ...
69               ts-config-mongo      socket     25      7.000000      7.000000      7.000000      7.000000      7.000000
70             ts-config-service         cpu     25      1.003346      1.325006      1.308087      1.535716      1.479208
71             ts-config-service      diskio     25  35201.265363  53029.475539  52983.734182  73011.792117  63290.758159
72             ts-config-service  latency-50     25      0.003311      0.003472      0.003462      0.003568      0.003543
57              ts-basic-service  latency-50     25      0.008031      0.008103      0.008108      0.008199      0.008189

[50 rows x 8 columns],                 cmdb_id                                       trace_name  count       min     median       mean        max        p95
3                  root   trace.to_ts-admin-basic-info-service.row_count     25  3.000000  11.000000  10.800000  18.000000  16.800000
824  ts-travel2-service      trace.from_ts-travel2-service.duration_mean     25  0.008870   0.010538   0.010807   0.015234   0.013002
825  ts-travel2-service       trace.from_ts-travel2-service.duration_p95     25  0.043810   0.046381   0.049311   0.063873   0.061474
826  ts-travel2-service         trace.from_ts-travel2-service.error_rate     25  0.000000   0.000000   0.000000   0.000000   0.000000
844  ts-travel2-service          trace.to_ts-train-service.duration_mean     25  0.003834   0.004110   0.004141   0.004509   0.004434
..                  ...                                              ...    ...       ...        ...        ...        ...        ...
45                 root  trace.to_ts-inside-payment-service.duration_p95     25  0.007253   0.037810   0.034307   0.055216   0.053654
46                 root    trace.to_ts-inside-payment-service.error_rate     25  0.000000   0.000000   0.000000   0.000000   0.000000
47                 root     trace.to_ts-inside-payment-service.row_count     25  2.000000  11.000000  11.320000  23.000000  21.800000
96                 root         trace.to_ts-travel-service.duration_mean     25  0.016949   0.023667   0.023025   0.030362   0.028112
97                 root          trace.to_ts-travel-service.duration_p95     25  0.045728   0.054280   0.059348   0.079812   0.078734

[50 rows x 8 columns],                         cmdb_id         log_name  count  min  median      mean   max     p95
0   ts-admin-basic-info-service  log.error_count     25    0     0.0     0.000     0     0.0
1   ts-admin-basic-info-service    log.row_count     25    6    30.0    27.920    32    32.0
13             ts-basic-service    log.row_count     25  535  2443.0  2362.400  2840  2663.4
12             ts-basic-service  log.error_count     25    0     0.0     0.000     0     0.0
11              ts-auth-service    log.row_count     25   52   328.0   315.360   356   347.6
..                          ...              ...    ...  ...     ...       ...   ...     ...
2       ts-admin-travel-service  log.error_count     24    0     0.0     0.000     0     0.0
21           ts-consign-service    log.row_count     24    7    42.0    39.375    70    63.0
20           ts-consign-service  log.error_count     24    0     0.0     0.000     0     0.0
39        ts-notification-mongo    log.row_count     24    1     1.0     1.000     1     1.0
38        ts-notification-mongo  log.error_count     24    0     0.0     0.000     0     0.0

[50 rows x 8 columns],                          cmdb_id
0                           root
1    ts-admin-basic-info-service
2         ts-admin-order-service
3         ts-admin-route-service
4        ts-admin-travel-service
..                           ...
64                 ts-user-mongo
65               ts-user-service
66  ts-verification-code-service
67              ts-voucher-mysql
68            ts-voucher-service

[69 rows x 1 columns],                   timestamp                      cmdb_id    kpi_name         value
0 2024-01-24 08:21:00+00:00  ts-admin-basic-info-service         cpu      0.391499
1 2024-01-24 08:21:00+00:00  ts-admin-basic-info-service      diskio  63980.836367
2 2024-01-24 08:21:00+00:00  ts-admin-basic-info-service       error      0.000000
3 2024-01-24 08:21:00+00:00  ts-admin-basic-info-service  latency-50      0.016897
4 2024-01-24 08:21:00+00:00  ts-admin-basic-info-service  latency-90      0.023379,                   timestamp cmdb_id                                         trace_name     value
0 2024-01-24 08:21:00+00:00    root  trace.to_ts-admin-basic-info-service.duration_...  0.008776
1 2024-01-24 08:21:00+00:00    root  trace.to_ts-admin-basic-info-service.duration_p95  0.010154
2 2024-01-24 08:21:00+00:00    root    trace.to_ts-admin-basic-info-service.error_rate  0.000000
3 2024-01-24 08:21:00+00:00    root     trace.to_ts-admin-basic-info-service.row_count  3.000000
4 2024-01-24 08:21:00+00:00    root             trace.to_ts-auth-service.duration_mean  0.048700,                   timestamp                      cmdb_id         log_name  value
0 2024-01-24 08:21:00+00:00  ts-admin-basic-info-service  log.error_count      0
1 2024-01-24 08:21:00+00:00  ts-admin-basic-info-service    log.row_count      6
2 2024-01-24 08:21:00+00:00           ts-assurance-mongo  log.error_count      0
3 2024-01-24 08:21:00+00:00           ts-assurance-mongo    log.row_count      1
4 2024-01-24 08:21:00+00:00              ts-auth-service  log.error_count      0, Empty DataFrame
Columns: [timestamp, cmdb_id, message, error]
Index: [])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import numpy as np
import pandas as pd

# Reuse metric_df already in session; ensure timestamp parsed per rules
metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# 1) Compute global P95 and P5 per (cmdb_id, kpi_name) over the full series
thresholds = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'])['value']
    .agg(p95=lambda x: x.quantile(0.95), p5=lambda x: x.quantile(0.05))
    .reset_index()
)

# 2) Filter metric.csv to the incident window (inclusive)
start = pd.to_datetime('2024-01-24 08:18:50', utc=True)
end = pd.to_datetime('2024-01-24 08:48:50', utc=True)
metric_filtered = metric_df.loc[
    (metric_df['timestamp'] >= start) & (metric_df['timestamp'] <= end),
    ['timestamp', 'cmdb_id', 'kpi_name', 'value']
].copy()

# 3) Join filtered rows with P95 and P5
merged = metric_filtered.merge(thresholds, on=['cmdb_id', 'kpi_name'], how='left')

# 4) High anomalies: value >= p95, compute severity = value / p95
high = merged.loc[merged['value'] >= merged['p95']].copy()
# handle p95 == 0: set severity to inf when value>0, else NaN
high['severity'] = np.where(
    high['p95'] > 0,
    high['value'] / high['p95'],
    np.where(high['value'] > 0, np.inf, np.nan)
)
high_top = (
    high
    .loc[:, ['timestamp', 'cmdb_id', 'kpi_name', 'value', 'p95', 'severity']]
    .sort_values('severity', ascending=False)
    .head(20)
    .reset_index(drop=True)
)

# 5) Low anomalies: value <= p5, compute severity_low = p5 / max(value, 1e-12)
low = merged.loc[merged['value'] <= merged['p5']].copy()
low['severity_low'] = low['p5'] / np.maximum(low['value'], 1e-12)
low_top = (
    low
    .loc[:, ['timestamp', 'cmdb_id', 'kpi_name', 'value', 'p5', 'severity_low']]
    .sort_values('severity_low', ascending=False)
    .head(20)
    .reset_index(drop=True)
)

# 6) Aggregate anomaly counts per cmdb_id (count of high + low anomalies), top 20
# Concatenate high and low, treat each row as one anomaly occurrence
anomalies_concat = pd.concat([high.loc[:, ['timestamp','cmdb_id','kpi_name','value','severity']],
                              low.loc[:, ['timestamp','cmdb_id','kpi_name','value','severity_low']]],
                             ignore_index=True, sort=False)

agg_counts = (
    anomalies_concat
    .groupby('cmdb_id', dropna=False)
    .agg(anomaly_count=('cmdb_id', 'size'),
         earliest_anomaly=('timestamp', 'min'))
    .reset_index()
    .sort_values('anomaly_count', ascending=False)
    .head(20)
    .reset_index(drop=True)
)

# 7) First 10 rows of the filtered metric data for quick inspection
filtered_head = metric_filtered.head(10).reset_index(drop=True)

# Return the requested compact outputs
high_top, low_top, agg_counts, filtered_head

```
Out[2]:
```
summary = (
    "Analysis method: computed global P95 and P5 per (cmdb_id, kpi_name) over the entire metric series, "
    "then filtered metric rows inside the incident window 2024-01-24 08:18:50–08:48:50 UTC and compared values to those thresholds.\n\n"
    "Key findings:\n"
    "- Top high anomalies (value >= P95), strongest severities (value / p95):\n"
    "  1) ts-contacts-service — cpu at 2024-01-24 08:21:00 — severity ≈ 2.50 (value ≈ 1.637 vs p95 ≈ 0.656)\n"
    "  2) ts-user-service — cpu at 2024-01-24 08:27:00 — severity ≈ 2.42\n"
    "  3) ts-consign-price-service — latency-90 at 08:26 — severity ≈ 2.27\n"
    "  4) ts-price-service — cpu at 08:24 — severity ≈ 1.90\n"
    "  5) ts-consign-service — diskio at 08:35 — severity ≈ 1.70\n"
    "  (Other notable disk I/O high anomalies: ts-auth-service, ts-travel2-service, ts-basic-service, ts-execute-service, etc.)\n\n"
    "- Top low anomalies (value <= P5), strongest severities (p5 / value):\n"
    "  * ts-preserve-other-service — workload had zero or near-zero values (severity extremely large due to value≈0 vs p5>0).\n"
    "  * Several workload metrics (ts-consign-service, ts-contacts-service, ts-food-service, ts-user-service, ts-payment-service) show significant drops below their P5 thresholds.\n"
    "  * Some diskio values are notably below P5 for certain mongo or service components.\n\n"
    "- Anomaly distribution (counts):\n"
    "  * Large counts concentrated in many backend components (top examples): ts-voucher-mysql and ts-food-mysql (154 anomalies each), ui-dashboard (124), avatar-service, ticket-office-service, ts-voucher-service (≈108 each), and several admin/mongo components. These cmdb_ids show the largest number of threshold-crossing records within the incident window.\n\n"
    "- Quick data inspection: first rows of the filtered window show typical metrics for ts-admin-basic-info-service at 2024-01-24 08:21:00.\n\n"
    "Interpretation / next steps:\n"
    "- Multiple services show CPU and disk I/O spikes above their historical P95s, and several services show steep workload drops (value <= P5). High disk I/O and CPU spikes across services suggest platform-level stress or a shared resource issue; the many anomalies clustered on mysql/mongo components point to heavy database activity or contention.\n"
    "- Recommended next actions: examine database (mysql/mongo) health and I/O metrics, check network/disk subsystem for contention, review service logs and traces for correlated errors or timeouts around 08:21–08:35 UTC, and prioritize investigation on the components with highest counts (voucher-mysql, food-mysql, ui-dashboard, ticket-office-service, voucher-service).\n"
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(                   timestamp                       cmdb_id    kpi_name          value            p95  severity
0  2024-01-24 08:21:00+00:00           ts-contacts-service         cpu       1.637223       0.655672  2.497014
1  2024-01-24 08:27:00+00:00               ts-user-service         cpu       1.020606       0.421279  2.422638
2  2024-01-24 08:26:00+00:00      ts-consign-price-service  latency-90       0.051279       0.022605  2.268488
3  2024-01-24 08:24:00+00:00              ts-price-service         cpu       2.426752       1.274692  1.903795
4  2024-01-24 08:26:00+00:00      ts-consign-price-service         cpu       1.193999       0.651569  1.832499
5  2024-01-24 08:35:00+00:00            ts-consign-service      diskio  327600.792929  192392.099361  1.702777
6  2024-01-24 08:25:00+00:00           ts-security-service  latency-90       0.138750       0.082967  1.672358
7  2024-01-24 08:35:00+00:00               ts-auth-service      diskio  238490.757428  153336.114528  1.555346
8  2024-01-24 08:21:00+00:00            ts-travel2-service      diskio   91482.697824   58996.996300  1.550633
9  2024-01-24 08:22:00+00:00          ts-assurance-service         cpu       0.753213       0.495179  1.521095
10 2024-01-24 08:21:00+00:00              ts-basic-service      diskio   92437.857458   61711.261508  1.497909
11 2024-01-24 08:25:00+00:00     ts-inside-payment-service         cpu       1.214090       0.827671  1.466875
12 2024-01-24 08:21:00+00:00            ts-execute-service      diskio   87260.332339   60358.410302  1.445703
13 2024-01-24 08:30:00+00:00      ts-ticket-office-service         cpu       0.019060       0.013266  1.436720
14 2024-01-24 08:21:00+00:00            ts-payment-service    workload       0.333000       0.241847  1.376905
15 2024-01-24 08:27:00+00:00           ts-contacts-service  latency-90       0.041350       0.030882  1.338960
16 2024-01-24 08:35:00+00:00  ts-verification-code-service      diskio  107117.885758   80948.130400  1.323290
17 2024-01-24 08:45:00+00:00     ts-inside-payment-service  latency-90       0.154059       0.116604  1.321211
18 2024-01-24 08:21:00+00:00        ts-admin-route-service      diskio   67375.999153   51109.998689  1.318255
19 2024-01-24 08:35:00+00:00            ts-station-service      diskio  191455.214698  146457.885243  1.307237,                    timestamp                    cmdb_id    kpi_name          value             p5  severity_low
0  2024-01-24 08:27:00+00:00  ts-preserve-other-service    workload       0.000000       0.012250  1.225000e+10
1  2024-01-24 08:44:00+00:00         ts-consign-service    workload       0.053200       0.099800  1.875940e+00
2  2024-01-24 08:21:00+00:00  ts-preserve-other-service    workload       0.006700       0.012250  1.828358e+00
3  2024-01-24 08:44:00+00:00   ts-consign-price-service    workload       0.062250       0.098683  1.585274e+00
4  2024-01-24 08:28:00+00:00        ts-contacts-service    workload       0.211133       0.322197  1.526034e+00
5  2024-01-24 08:23:00+00:00            ts-food-service    workload       0.455417       0.656763  1.442115e+00
6  2024-01-24 08:45:00+00:00         ts-execute-service      diskio   32825.164033   47294.498151  1.440800e+00
7  2024-01-24 08:21:00+00:00     ts-ticket-office-mongo      diskio  198266.988664  284733.136052  1.436110e+00
8  2024-01-24 08:28:00+00:00        ts-security-service    workload       0.104467       0.148147  1.418124e+00
9  2024-01-24 08:28:00+00:00            ts-user-service    workload       0.108933       0.148943  1.367289e+00
10 2024-01-24 08:38:00+00:00          ts-config-service      diskio   35201.265363   45878.795454  1.303328e+00
11 2024-01-24 08:43:00+00:00         ts-payment-service    workload       0.027733       0.035520  1.280769e+00
12 2024-01-24 08:44:00+00:00         ts-consign-service         cpu       0.133747       0.171136  1.279549e+00
13 2024-01-24 08:43:00+00:00         ts-payment-service         cpu       0.126380       0.159990  1.265949e+00
14 2024-01-24 08:21:00+00:00       ts-assurance-service      diskio   29168.661923   36589.826126  1.254423e+00
15 2024-01-24 08:36:00+00:00   ts-consign-price-service  latency-90       0.013525       0.016819  1.243534e+00
16 2024-01-24 08:39:00+00:00      ts-ticketinfo-service      diskio   36418.652083   45070.992247  1.237580e+00
17 2024-01-24 08:44:00+00:00   ts-consign-price-service         cpu       0.141125       0.174110  1.233732e+00
18 2024-01-24 08:29:00+00:00      ts-admin-user-service      diskio   27845.552585   34280.410803  1.231091e+00
19 2024-01-24 08:39:00+00:00           ts-basic-service      socket       8.016667       9.743333  1.215385e+00,                         cmdb_id  anomaly_count          earliest_anomaly
0              ts-voucher-mysql            154 2024-01-24 08:21:00+00:00
1                 ts-food-mysql            154 2024-01-24 08:21:00+00:00
2               ts-ui-dashboard            124 2024-01-24 08:21:00+00:00
3             ts-avatar-service            108 2024-01-24 08:21:00+00:00
4      ts-ticket-office-service            108 2024-01-24 08:21:00+00:00
5            ts-voucher-service            108 2024-01-24 08:21:00+00:00
6       ts-admin-travel-service            100 2024-01-24 08:21:00+00:00
7   ts-admin-basic-info-service             78 2024-01-24 08:21:00+00:00
8               ts-food-service             78 2024-01-24 08:21:00+00:00
9               ts-auth-service             78 2024-01-24 08:21:00+00:00
10           ts-consign-service             78 2024-01-24 08:21:00+00:00
11            ts-security-mongo             62 2024-01-24 08:21:00+00:00
12               ts-route-mongo             62 2024-01-24 08:21:00+00:00
13               ts-order-mongo             62 2024-01-24 08:21:00+00:00
14             ts-payment-mongo             62 2024-01-24 08:21:00+00:00
15         ts-order-other-mongo             62 2024-01-24 08:21:00+00:00
16        ts-notification-mongo             62 2024-01-24 08:21:00+00:00
17       ts-consign-price-mongo             62 2024-01-24 08:21:00+00:00
18             ts-consign-mongo             62 2024-01-24 08:21:00+00:00
19           ts-assurance-mongo             62 2024-01-24 08:21:00+00:00,                   timestamp                      cmdb_id    kpi_name         value
0 2024-01-24 08:21:00+00:00  ts-admin-basic-info-service         cpu  3.914995e-01
1 2024-01-24 08:21:00+00:00  ts-admin-basic-info-service      diskio  6.398084e+04
2 2024-01-24 08:21:00+00:00  ts-admin-basic-info-service       error  0.000000e+00
3 2024-01-24 08:21:00+00:00  ts-admin-basic-info-service  latency-50  1.689743e-02
4 2024-01-24 08:21:00+00:00  ts-admin-basic-info-service  latency-90  2.337949e-02
5 2024-01-24 08:21:00+00:00  ts-admin-basic-info-service         mem  2.326667e+08
6 2024-01-24 08:21:00+00:00  ts-admin-basic-info-service      socket  6.200000e+00
7 2024-01-24 08:21:00+00:00  ts-admin-basic-info-service    workload  7.333000e-01
8 2024-01-24 08:21:00+00:00       ts-admin-order-service         cpu  1.116891e-01
9 2024-01-24 08:21:00+00:00       ts-admin-order-service      diskio  5.055656e+04)```
```

In [ ]:
# In[3]:

import numpy as np
import pandas as pd

# Reuse existing metric_df from session; ensure correct parsing and types
metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# Services to restrict analysis to
services = ['ts-auth-service','ts-order-service','ts-route-service','ts-train-service','ts-travel-service']

# 1) Compute global P95 and P5 for each (cmdb_id, kpi_name) using the full series (restricted to chosen services)
thresholds_sub = (
    metric_df[metric_df['cmdb_id'].isin(services)]
    .groupby(['cmdb_id', 'kpi_name'])['value']
    .agg(p95=lambda x: x.quantile(0.95), p5=lambda x: x.quantile(0.05))
    .reset_index()
)

# 2) Filter metric.csv to the incident window (inclusive)
start = pd.to_datetime('2024-01-24 08:18:50', utc=True)
end = pd.to_datetime('2024-01-24 08:48:50', utc=True)
metric_filtered_sub = metric_df.loc[
    (metric_df['timestamp'] >= start) &
    (metric_df['timestamp'] <= end) &
    (metric_df['cmdb_id'].isin(services)),
    ['timestamp', 'cmdb_id', 'kpi_name', 'value']
].copy()

# 3) Join filtered rows with their global P95 and P5
merged_sub = metric_filtered_sub.merge(thresholds_sub, on=['cmdb_id', 'kpi_name'], how='left')

# 4) High anomalies: value >= p95, severity = value / p95
high_sub = merged_sub.loc[merged_sub['value'] >= merged_sub['p95']].copy()
high_sub['severity'] = np.where(
    high_sub['p95'] > 0,
    high_sub['value'] / high_sub['p95'],
    np.where(high_sub['value'] > 0, np.inf, np.nan)
)
high_top = (
    high_sub
    .loc[:, ['timestamp', 'cmdb_id', 'kpi_name', 'value', 'p95', 'severity']]
    .sort_values('severity', ascending=False)
    .head(20)
    .reset_index(drop=True)
)

# 5) Low anomalies: value <= p5, severity_low = p5 / max(value, 1e-12)
low_sub = merged_sub.loc[merged_sub['value'] <= merged_sub['p5']].copy()
low_sub['severity_low'] = low_sub['p5'] / np.maximum(low_sub['value'], 1e-12)
low_top = (
    low_sub
    .loc[:, ['timestamp', 'cmdb_id', 'kpi_name', 'value', 'p5', 'severity_low']]
    .sort_values('severity_low', ascending=False)
    .head(20)
    .reset_index(drop=True)
)

# 6) Aggregate table per cmdb_id with anomaly_count, earliest_anomaly, and top-3 anomalous kpi_name by count
# Concatenate high and low anomalies for counting (keep kpi_name and timestamp)
anoms = pd.concat([
    high_sub.loc[:, ['timestamp', 'cmdb_id', 'kpi_name']],
    low_sub.loc[:, ['timestamp', 'cmdb_id', 'kpi_name']]
], ignore_index=True, sort=False)

# If no anomalies for some services, ensure they still appear with zero counts
anom_counts = (
    anoms.groupby('cmdb_id', dropna=False)
    .agg(anomaly_count=('kpi_name', 'size'),
         earliest_anomaly=('timestamp', 'min'))
    .reset_index()
)

# Ensure all requested services are present
all_services_df = pd.DataFrame({'cmdb_id': services})
agg_full = all_services_df.merge(anom_counts, on='cmdb_id', how='left')
agg_full['anomaly_count'] = agg_full['anomaly_count'].fillna(0).astype(int)
agg_full['earliest_anomaly'] = agg_full['earliest_anomaly']  # NaT if none

# Compute top-3 anomalous kpi_name by count per cmdb_id
kpi_counts = (
    anoms.groupby(['cmdb_id', 'kpi_name'])
    .size()
    .reset_index(name='count')
)

def top3_kpis_for_cmdb(cmdb):
    sub = kpi_counts.loc[kpi_counts['cmdb_id'] == cmdb]
    if sub.empty:
        return ""
    top = sub.sort_values('count', ascending=False).head(3)
    # format as "kpi1(count),kpi2(count),..."
    return ", ".join(f"{row.kpi_name}({int(row['count'])})" for _, row in top.iterrows())

agg_full['top3_kpis_by_count'] = agg_full['cmdb_id'].map(top3_kpis_for_cmdb)

# Final aggregate result limited to these five services
agg_result = agg_full[['cmdb_id', 'anomaly_count', 'earliest_anomaly', 'top3_kpis_by_count']]

# 7) First 10 rows of the filtered metric data within the incident window for these services
filtered_head = metric_filtered_sub.head(10).reset_index(drop=True)

# Return the compact outputs
high_top, low_top, agg_result, filtered_head

```
Out[3]:
```
summary = (
    "Summary (restricted to ts-auth-service, ts-order-service, ts-route-service, ts-train-service, ts-travel-service):\n\n"
    "1) High anomalies (value >= global P95) — top examples by severity:\n"
    "- ts-auth-service: diskio at 2024-01-24 08:35 UTC (severity ≈ 1.56), latency-90 at 08:36 (≈1.20), cpu at 08:36 (≈1.16).\n"
    "- ts-order-service: diskio at 08:45 (≈1.16), workload and cpu moments slightly above P95.\n"
    "- ts-travel-service: latency-90 at 08:43 (≈1.15).\n"
    "- ts-train-service and ts-route-service show smaller CPU/diskio/latency spikes (severity ≈1.02–1.06).\n\n"
    "2) Low anomalies (value <= global P5) — top examples:\n"
    "- ts-order-service: latency-90 at 08:22 UTC is below its P5 (severity_low ≈ 1.14); cpu and workload had points below P5 as well.\n"
    "- ts-travel-service and ts-auth-service have a few metrics below P5 (workload/latency/socket), but severities are modest (~1.03–1.13).\n\n"
    "3) Anomaly counts and hotspots (within the incident window):\n"
    "- ts-auth-service: highest anomaly_count = 78, earliest anomaly at 2024-01-24 08:21 UTC. Top anomalous KPIs: error(50), cpu(4), diskio(4).\n"
    "- ts-route-service: anomaly_count = 29, top KPIs: socket(5), diskio(4), cpu(4).\n"
    "- ts-order-service, ts-train-service, ts-travel-service: each ~28 anomalies; top KPIs for these are cpu, diskio, latency-50 (counts ≈4 each).\n\n"
    "4) Interpretation and recommended next steps:\n"
    "- ts-auth-service is the dominant outlier (many 'error' metric anomalies plus CPU and disk I/O spikes). Investigate auth service logs and traces around 08:21–08:36 UTC.\n"
    "- Disk I/O and CPU spikes also appear across order/train/travel/route services (but with smaller severities), suggesting either heavy workload or shared resource contention (DB, disk, or network).\n"
    "- Recommended actions: check ts-auth-service logs for errors, inspect DB/disk subsystem and network I/O for contention, and correlate with traces/slow requests during 08:21–08:45 UTC.\n\n"
    "5) Quick data preview: filtered metrics head shows ts-auth-service rows at 08:21 (cpu≈13.3, diskio≈59176, error=0, latency-50≈0.214, socket=11).\n\n"
    "In short: prioritize investigating ts-auth-service (errors + resource spikes) and underlying shared storage/DB I/O as likely contributors."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(                   timestamp            cmdb_id    kpi_name         value           p95  severity
0  2024-01-24 08:35:00+00:00    ts-auth-service      diskio  2.384908e+05  1.533361e+05  1.555346
1  2024-01-24 08:36:00+00:00    ts-auth-service  latency-90  1.309063e+00  1.089282e+00  1.201767
2  2024-01-24 08:36:00+00:00    ts-auth-service         cpu  1.785901e+01  1.535322e+01  1.163209
3  2024-01-24 08:45:00+00:00   ts-order-service      diskio  2.902872e+05  2.506511e+05  1.158132
4  2024-01-24 08:43:00+00:00  ts-travel-service  latency-90  3.210907e-01  2.795632e-01  1.148544
5  2024-01-24 08:44:00+00:00   ts-order-service    workload  7.982283e+00  7.449213e+00  1.071561
6  2024-01-24 08:44:00+00:00    ts-auth-service  latency-50  2.902226e-01  2.723736e-01  1.065531
7  2024-01-24 08:36:00+00:00   ts-train-service      diskio  6.403853e+04  6.027593e+04  1.062423
8  2024-01-24 08:45:00+00:00   ts-order-service  latency-50  1.044636e-02  9.857240e-03  1.059765
9  2024-01-24 08:44:00+00:00   ts-route-service  latency-90  6.415475e-02  6.111716e-02  1.049701
10 2024-01-24 08:44:00+00:00   ts-train-service         cpu  3.526004e+00  3.376696e+00  1.044217
11 2024-01-24 08:25:00+00:00   ts-order-service  latency-90  2.474221e-02  2.379467e-02  1.039822
12 2024-01-24 08:44:00+00:00   ts-order-service         cpu  2.257571e+00  2.186388e+00  1.032557
13 2024-01-24 08:42:00+00:00   ts-route-service      diskio  1.932977e+10  1.873408e+10  1.031797
14 2024-01-24 08:40:00+00:00  ts-travel-service         cpu  3.624850e+00  3.535619e+00  1.025238
15 2024-01-24 08:32:00+00:00   ts-order-service      socket  1.275000e+01  1.249667e+01  1.020272
16 2024-01-24 08:42:00+00:00    ts-auth-service      diskio  1.563964e+05  1.533361e+05  1.019958
17 2024-01-24 08:44:00+00:00   ts-train-service    workload  2.305562e+01  2.264192e+01  1.018271
18 2024-01-24 08:44:00+00:00   ts-route-service  latency-50  4.316980e-03  4.240996e-03  1.017917
19 2024-01-24 08:44:00+00:00   ts-order-service      diskio  2.544840e+05  2.506511e+05  1.015291,                    timestamp            cmdb_id    kpi_name         value            p5  severity_low
0  2024-01-24 08:22:00+00:00   ts-order-service  latency-90      0.016126      0.018342      1.137416
1  2024-01-24 08:37:00+00:00   ts-order-service         cpu      1.166210      1.316589      1.128946
2  2024-01-24 08:36:00+00:00   ts-order-service    workload      3.999883      4.490093      1.122556
3  2024-01-24 08:36:00+00:00  ts-travel-service    workload      6.146550      6.874027      1.118355
4  2024-01-24 08:45:00+00:00  ts-travel-service         cpu      2.112956      2.335632      1.105386
5  2024-01-24 08:33:00+00:00    ts-auth-service      diskio  39596.800418  43694.260284      1.103480
6  2024-01-24 08:28:00+00:00  ts-travel-service  latency-90      0.080450      0.087768      1.090960
7  2024-01-24 08:24:00+00:00  ts-travel-service      socket     14.116667     15.073333      1.067769
8  2024-01-24 08:21:00+00:00    ts-auth-service  latency-90      0.427143      0.455254      1.065811
9  2024-01-24 08:21:00+00:00    ts-auth-service      socket     11.000000     11.600588      1.054599
10 2024-01-24 08:26:00+00:00   ts-train-service         cpu      2.629995      2.772153      1.054053
11 2024-01-24 08:24:00+00:00   ts-route-service         cpu      3.684415      3.882834      1.053853
12 2024-01-24 08:23:00+00:00   ts-order-service      diskio  41560.712469  43392.512764      1.044075
13 2024-01-24 08:23:00+00:00   ts-order-service      socket     10.616667     11.000000      1.036107
14 2024-01-24 08:36:00+00:00   ts-train-service    workload     18.471250     19.099097      1.033990
15 2024-01-24 08:22:00+00:00    ts-auth-service  latency-90      0.440495      0.455254      1.033505
16 2024-01-24 08:25:00+00:00   ts-order-service    workload      4.346783      4.490093      1.032969
17 2024-01-24 08:25:00+00:00  ts-travel-service  latency-50      0.015488      0.015954      1.030090
18 2024-01-24 08:36:00+00:00  ts-travel-service         cpu      2.268371      2.335632      1.029651
19 2024-01-24 08:25:00+00:00  ts-travel-service  latency-90      0.085320      0.087768      1.028685,              cmdb_id  anomaly_count          earliest_anomaly                top3_kpis_by_count
0    ts-auth-service             78 2024-01-24 08:21:00+00:00      error(50), cpu(4), diskio(4)
1   ts-order-service             28 2024-01-24 08:21:00+00:00  cpu(4), diskio(4), latency-50(4)
2   ts-route-service             29 2024-01-24 08:21:00+00:00      socket(5), diskio(4), cpu(4)
3   ts-train-service             28 2024-01-24 08:21:00+00:00  cpu(4), diskio(4), latency-50(4)
4  ts-travel-service             28 2024-01-24 08:21:00+00:00  cpu(4), diskio(4), latency-50(4),                   timestamp           cmdb_id    kpi_name         value
0 2024-01-24 08:21:00+00:00   ts-auth-service         cpu  1.331145e+01
1 2024-01-24 08:21:00+00:00   ts-auth-service      diskio  5.917620e+04
2 2024-01-24 08:21:00+00:00   ts-auth-service       error  0.000000e+00
3 2024-01-24 08:21:00+00:00   ts-auth-service  latency-50  2.141875e-01
4 2024-01-24 08:21:00+00:00   ts-auth-service  latency-90  4.271429e-01
5 2024-01-24 08:21:00+00:00   ts-auth-service         mem  2.446295e+08
6 2024-01-24 08:21:00+00:00   ts-auth-service      socket  1.100000e+01
7 2024-01-24 08:21:00+00:00   ts-auth-service    workload  2.653600e+00
8 2024-01-24 08:21:00+00:00  ts-order-service         cpu  1.740794e+00
9 2024-01-24 08:21:00+00:00  ts-order-service      diskio  4.309626e+04)```
```

In [ ]:
# In[4]:

import re
import numpy as np
import pandas as pd

# Ensure trace_df is parsed and numeric (reuse from session)
trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')

# Target services
services = ['ts-auth-service','ts-order-service','ts-route-service','ts-train-service','ts-travel-service']

# 1) Global P95 and P5 per (cmdb_id, trace_name) using full series
trace_thresholds = (
    trace_df
    .groupby(['cmdb_id', 'trace_name'])['value']
    .agg(p95=lambda x: x.quantile(0.95), p5=lambda x: x.quantile(0.05))
    .reset_index()
)

# 2) Filter trace.csv to the incident window (inclusive)
start = pd.to_datetime('2024-01-24 08:18:50', utc=True)
end = pd.to_datetime('2024-01-24 08:48:50', utc=True)
trace_window = trace_df.loc[(trace_df['timestamp'] >= start) & (trace_df['timestamp'] <= end)].copy()

# 3) From filtered traces, keep only rows where trace_name contains any service identifier OR cmdb_id is one of the five
pattern = '|'.join(map(re.escape, services))
mask = trace_window['trace_name'].str.contains(pattern, na=False) | trace_window['cmdb_id'].isin(services)
trace_filtered = trace_window.loc[mask, ['timestamp', 'cmdb_id', 'trace_name', 'value']].copy()

# 4) Join filtered rows with their global P95/P5
trace_merged = trace_filtered.merge(trace_thresholds, on=['cmdb_id', 'trace_name'], how='left')

# Helper to detect feature type
def detect_feature(name):
    for f in ['duration_mean', 'duration_p95', 'error_rate', 'row_count']:
        if f in name:
            return f
    return 'other'

trace_merged['feature'] = trace_merged['trace_name'].apply(detect_feature)

# 4a) High-side anomalies for features in ['duration_mean','duration_p95','error_rate'] where value >= p95
high_mask = trace_merged['feature'].isin(['duration_mean', 'duration_p95', 'error_rate']) & (trace_merged['value'] >= trace_merged['p95'])
high = trace_merged.loc[high_mask].copy()
high['severity'] = np.where(
    high['p95'] > 0,
    high['value'] / high['p95'],
    np.where(high['value'] > 0, np.inf, np.nan)
)
high_top = high.loc[:, ['timestamp', 'cmdb_id', 'trace_name', 'value', 'p95', 'severity']].sort_values('severity', ascending=False).head(20).reset_index(drop=True)

# 4b) Low-side anomalies for feature 'row_count' where value <= p5
low_mask = (trace_merged['feature'] == 'row_count') & (trace_merged['value'] <= trace_merged['p5'])
low = trace_merged.loc[low_mask].copy()
low['severity_low'] = trace_merged.loc[low_mask, 'p5'] / np.maximum(low['value'], 1e-12)
low_top = low.loc[:, ['timestamp', 'cmdb_id', 'trace_name', 'value', 'p5', 'severity_low']].sort_values('severity_low', ascending=False).head(20).reset_index(drop=True)

# 5) Aggregate table for the five services
anoms = pd.concat([
    high.loc[:, ['timestamp', 'cmdb_id', 'feature']],
    low.loc[:, ['timestamp', 'cmdb_id', 'feature']]
], ignore_index=True, sort=False)

# Count and earliest anomaly per cmdb_id
anom_counts = anoms.groupby('cmdb_id', dropna=False).agg(
    trace_anomaly_count=('feature', 'size'),
    earliest_trace_anomaly=('timestamp', 'min')
).reset_index()

# Ensure all five services are present
all_services_df = pd.DataFrame({'cmdb_id': services})
agg = all_services_df.merge(anom_counts, on='cmdb_id', how='left')
agg['trace_anomaly_count'] = agg['trace_anomaly_count'].fillna(0).astype(int)
agg['earliest_trace_anomaly'] = agg['earliest_trace_anomaly']  # NaT if none

# Top2 trace features by count per service
feature_counts = anoms.groupby(['cmdb_id', 'feature']).size().reset_index(name='count')
def top2_feats(cmdb):
    sub = feature_counts.loc[feature_counts['cmdb_id'] == cmdb]
    if sub.empty:
        return ""
    top = sub.sort_values('count', ascending=False).head(2)
    return ", ".join(f"{row.feature}({int(row['count'])})" for _, row in top.iterrows())

agg['top2_trace_features_by_count'] = agg['cmdb_id'].map(top2_feats)
agg_result = agg[['cmdb_id', 'trace_anomaly_count', 'earliest_trace_anomaly', 'top2_trace_features_by_count']]

# 6) First 10 rows of the filtered trace data used for anomaly detection
filtered_head = trace_filtered.head(10).reset_index(drop=True)

# Return compact outputs
high_top, low_top, agg_result, filtered_head

```
Out[4]:
```
Summary of trace analysis for the five candidate services (ts-auth-service, ts-order-service, ts-route-service, ts-train-service, ts-travel-service):

What was done
- Computed global P95 and P5 per (cmdb_id, trace_name) over the full trace series.
- Filtered traces to the incident window 2024-01-24 08:18:50–08:48:50 UTC.
- Kept traces referencing the five services (either as cmdb_id or in trace_name).
- Flagged high-side anomalies for duration_mean / duration_p95 / error_rate (value >= P95) and low-side anomalies for row_count (value <= P5).

Key findings
- Biggest high-severity trace anomaly:
  - ts-order-service at 2024-01-24 08:40:00 — extreme duration_p95 spike (value ≈ 0.405 s vs p95 ≈ 0.0147 s; severity ≈ 27.5). The same timestamp shows to_ts-order-service/from_ts-order-service duration_p95 and duration_mean elevated, indicating very large latency around order service calls at ~08:40.
- Other notable high-side trace spikes (severity > ~1.3):
  - ts-order-service <-> ts-travel interactions (e.g., 08:34 duration_p95 and duration_mean with severity ≈ 2.73).
  - ts-travel_service, ts-train_service, ts-route_service and ts-auth_service multiple duration_p95/mean anomalies with severity typically 1.3–1.7.
- Widespread high error_rate anomalies:
  - Aggregation shows error_rate dominates anomaly counts for all five services (e.g., ts-travel-service: error_rate count ≈ 271; ts-order-service: ≈181; ts-train/ts-route similar). This indicates many trace-level errors across services during the window.
- Low-side row_count anomalies:
  - Several row_count values are far below their P5 (severity_low up to ~4.12), e.g., root -> to_ts-route and to_ts-auth row_counts and from_root row_counts for services at 08:21 — implying dramatically reduced observed trace volumes compared with normal (or sampling drop).
- Anomaly counts (within the five services, in the window):
  - ts-travel-service: highest trace_anomaly_count ≈ 350 (earliest anomaly 08:21)
  - ts-order-service: ≈ 251
  - ts-train-service: ≈ 188
  - ts-route-service: ≈ 186
  - ts-auth-service: ≈ 93
  - Earliest anomalies for all are at ~2024-01-24 08:21:00 UTC.

Interpretation
- The most urgent signal is a severe latency spike involving ts-order-service at ~08:40 (very high duration_p95 relative to norm), indicating a major slow path or cascading delay for order-related requests.
- Simultaneously, many error_rate anomalies and reduced row_count (trace volume) are present across services, suggesting widespread failures or degraded request processing and/or sampling differences early in the window (~08:21) and continued instability later.
- The pattern (high latencies + many error_rate anomalies + lower row counts) suggests both request failures and slowed processing across service calls—likely an inter-service or shared dependency problem (e.g., DB, downstream service, network) that impacts order/travel interactions heavily.

Recommended next steps
- Immediately investigate ts-order-service around 08:34–08:40:
  - Inspect order service logs and traces for errors, timeouts, and the specific slow endpoints.
  - Correlate with ts-travel and downstream calls (the traces show order ↔ travel high latencies).
- Review error logs and trace error details for ts-travel and ts-train (high error_rate counts).
- Check shared dependencies (databases, caches, network/disk I/O) for contention or failures around 08:21 and 08:34–08:40.
- Use full trace spans to trace root cause hop-by-hop for the ts-order spike (look for where latency increased first).

Quick peek at filtered traces (example rows)
- The filtered head shows standard to_ts- and from_ts- traces at 08:21 for these services (duration_mean/p95/row_count), confirming the traces are present and that the earliest anomalies start at ~08:21.

In short: prioritize investigating ts-order-service latency spike at ~08:40 and broader error_rate and trace-volume drops that affect ts-travel and related services, while checking shared dependencies.

The original code execution output of IPython Kernel is also provided below for reference:

(                   timestamp             cmdb_id                                  trace_name     value       p95   severity
0  2024-01-24 08:40:00+00:00    ts-order-service    trace.from_ts-order-service.duration_p95  0.405383  0.014715  27.548138
1  2024-01-24 08:40:00+00:00    ts-order-service      trace.to_ts-order-service.duration_p95  0.405383  0.014715  27.548138
2  2024-01-24 08:34:00+00:00    ts-order-service   trace.from_ts-travel-service.duration_p95  0.044984  0.016457   2.733379
3  2024-01-24 08:34:00+00:00   ts-travel-service      trace.to_ts-order-service.duration_p95  0.044984  0.016457   2.733379
4  2024-01-24 08:40:00+00:00    ts-order-service     trace.to_ts-order-service.duration_mean  0.035286  0.020599   1.712935
5  2024-01-24 08:40:00+00:00    ts-order-service   trace.from_ts-order-service.duration_mean  0.035286  0.020599   1.712935
6  2024-01-24 08:34:00+00:00   ts-travel-service     trace.to_ts-order-service.duration_mean  0.013284  0.009240   1.437737
7  2024-01-24 08:34:00+00:00    ts-order-service  trace.from_ts-travel-service.duration_mean  0.013284  0.009240   1.437737
8  2024-01-24 08:40:00+00:00  ts-travel2-service      trace.to_ts-train-service.duration_p95  0.008580  0.006107   1.404990
9  2024-01-24 08:40:00+00:00    ts-train-service  trace.from_ts-travel2-service.duration_p95  0.008580  0.006107   1.404990
10 2024-01-24 08:42:00+00:00   ts-travel-service     trace.to_ts-route-service.duration_mean  0.018340  0.013599   1.348560
11 2024-01-24 08:42:00+00:00    ts-route-service  trace.from_ts-travel-service.duration_mean  0.018340  0.013599   1.348560
12 2024-01-24 08:39:00+00:00   ts-travel-service       trace.to_ts-seat-service.duration_p95  0.109430  0.081232   1.347131
13 2024-01-24 08:39:00+00:00     ts-seat-service   trace.from_ts-travel-service.duration_p95  0.109430  0.081232   1.347131
14 2024-01-24 08:38:00+00:00     ts-food-service     trace.to_ts-travel-service.duration_p95  0.077466  0.057882   1.338356
15 2024-01-24 08:38:00+00:00   ts-travel-service     trace.from_ts-food-service.duration_p95  0.077466  0.057882   1.338356
16 2024-01-24 08:38:00+00:00     ts-auth-service     trace.from_ts-auth-service.duration_p95  0.085332  0.063904   1.335323
17 2024-01-24 08:38:00+00:00     ts-auth-service       trace.to_ts-auth-service.duration_p95  0.085332  0.063904   1.335323
18 2024-01-24 08:44:00+00:00    ts-route-service  trace.from_ts-travel2-service.duration_p95  0.056557  0.042482   1.331313
19 2024-01-24 08:44:00+00:00  ts-travel2-service      trace.to_ts-route-service.duration_p95  0.056557  0.042482   1.331313,                    timestamp             cmdb_id                               trace_name  value     p5  severity_low
0  2024-01-24 08:21:00+00:00                root      trace.to_ts-route-service.row_count  136.0  560.4      4.120588
1  2024-01-24 08:21:00+00:00    ts-route-service                trace.from_root.row_count  136.0  560.4      4.120588
2  2024-01-24 08:21:00+00:00     ts-auth-service                trace.from_root.row_count   21.0   86.4      4.114286
3  2024-01-24 08:21:00+00:00                root       trace.to_ts-auth-service.row_count   21.0   86.4      4.114286
4  2024-01-24 08:21:00+00:00    ts-route-service   trace.from_ts-travel-service.row_count    7.0   27.4      3.914286
5  2024-01-24 08:21:00+00:00   ts-travel-service      trace.to_ts-route-service.row_count    7.0   27.4      3.914286
6  2024-01-24 08:21:00+00:00    ts-train-service                trace.from_root.row_count  155.0  552.0      3.561290
7  2024-01-24 08:21:00+00:00                root      trace.to_ts-train-service.row_count  155.0  552.0      3.561290
8  2024-01-24 08:21:00+00:00    ts-train-service  trace.from_ts-travel2-service.row_count    7.0   22.0      3.142857
9  2024-01-24 08:21:00+00:00  ts-travel2-service      trace.to_ts-train-service.row_count    7.0   22.0      3.142857
10 2024-01-24 08:39:00+00:00   ts-travel-service      trace.to_ts-order-service.row_count    1.0    3.0      3.000000
11 2024-01-24 08:39:00+00:00    ts-order-service   trace.from_ts-travel-service.row_count    1.0    3.0      3.000000
12 2024-01-24 08:21:00+00:00  ts-travel2-service      trace.to_ts-route-service.row_count   19.0   56.2      2.957895
13 2024-01-24 08:21:00+00:00    ts-route-service  trace.from_ts-travel2-service.row_count   19.0   56.2      2.957895
14 2024-01-24 08:21:00+00:00    ts-order-service                trace.from_root.row_count   45.0  104.0      2.311111
15 2024-01-24 08:21:00+00:00                root      trace.to_ts-order-service.row_count   45.0  104.0      2.311111
16 2024-01-24 08:21:00+00:00    ts-route-service      trace.to_ts-route-service.row_count   70.0  158.4      2.262857
17 2024-01-24 08:21:00+00:00    ts-route-service    trace.from_ts-route-service.row_count   70.0  158.4      2.262857
18 2024-01-24 08:21:00+00:00    ts-train-service    trace.from_ts-train-service.row_count  112.0  218.6      1.951786
19 2024-01-24 08:21:00+00:00    ts-train-service      trace.to_ts-train-service.row_count  112.0  218.6      1.951786,              cmdb_id  trace_anomaly_count    earliest_trace_anomaly        top2_trace_features_by_count
0    ts-auth-service                   93 2024-01-24 08:21:00+00:00    error_rate(75), duration_mean(6)
1   ts-order-service                  251 2024-01-24 08:21:00+00:00      error_rate(181), row_count(42)
2   ts-route-service                  186 2024-01-24 08:21:00+00:00  error_rate(150), duration_mean(12)
3   ts-train-service                  188 2024-01-24 08:21:00+00:00      error_rate(150), row_count(14)
4  ts-travel-service                  350 2024-01-24 08:21:00+00:00      error_rate(271), row_count(35),                   timestamp cmdb_id                               trace_name      value
0 2024-01-24 08:21:00+00:00    root   trace.to_ts-auth-service.duration_mean   0.048700
1 2024-01-24 08:21:00+00:00    root    trace.to_ts-auth-service.duration_p95   0.214976
2 2024-01-24 08:21:00+00:00    root      trace.to_ts-auth-service.error_rate   0.000000
3 2024-01-24 08:21:00+00:00    root       trace.to_ts-auth-service.row_count  21.000000
4 2024-01-24 08:21:00+00:00    root  trace.to_ts-order-service.duration_mean   0.009223
5 2024-01-24 08:21:00+00:00    root   trace.to_ts-order-service.duration_p95   0.004262
6 2024-01-24 08:21:00+00:00    root     trace.to_ts-order-service.error_rate   0.000000
7 2024-01-24 08:21:00+00:00    root      trace.to_ts-order-service.row_count  45.000000
8 2024-01-24 08:21:00+00:00    root  trace.to_ts-route-service.duration_mean   0.002072
9 2024-01-24 08:21:00+00:00    root   trace.to_ts-route-service.duration_p95   0.003295)```
```

In [ ]:
# In[5]:

import numpy as np
import pandas as pd

# Reuse existing dataframes from session; ensure timestamps parsed per rules
log_df['timestamp'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True)
log_df['value'] = pd.to_numeric(log_df['value'], errors='coerce')
error_df['timestamp'] = pd.to_datetime(error_df['timestamp'], unit='s', utc=True)

# Target services
services = ['ts-auth-service','ts-order-service','ts-route-service','ts-train-service','ts-travel-service']

# 1) Global P95 and P5 per (cmdb_id, log_name) using full log series (no time filtering)
log_thresholds = (
    log_df
    .groupby(['cmdb_id', 'log_name'])['value']
    .agg(p95=lambda x: x.quantile(0.95), p5=lambda x: x.quantile(0.05))
    .reset_index()
)

# 2) Filter log.csv to the incident window (inclusive) and restrict to the five services
start = pd.to_datetime('2024-01-24 08:18:50', utc=True)
end = pd.to_datetime('2024-01-24 08:48:50', utc=True)
log_filtered = log_df.loc[
    (log_df['timestamp'] >= start) &
    (log_df['timestamp'] <= end) &
    (log_df['cmdb_id'].isin(services)),
    ['timestamp', 'cmdb_id', 'log_name', 'value']
].copy()

# 3) Join filtered rows with their global P95 and P5
log_merged = log_filtered.merge(log_thresholds, on=['cmdb_id', 'log_name'], how='left')

# 4) High-side log anomalies: log_name contains 'error' (case-insensitive) and value >= p95
err_mask = log_merged['log_name'].str.contains('error', case=False, na=False)
high_logs = log_merged.loc[err_mask & (log_merged['value'] >= log_merged['p95'])].copy()
high_logs['severity'] = np.where(
    high_logs['p95'] > 0,
    high_logs['value'] / high_logs['p95'],
    np.where(high_logs['value'] > 0, np.inf, np.nan)
)
high_top = (
    high_logs
    .loc[:, ['timestamp', 'cmdb_id', 'log_name', 'value', 'p95', 'severity']]
    .sort_values('severity', ascending=False)
    .head(20)
    .reset_index(drop=True)
)

# 5) Low-side log anomalies: log_name contains 'row_count' and value <= p5
row_mask = log_merged['log_name'].str.contains('row_count', case=False, na=False)
low_logs = log_merged.loc[row_mask & (log_merged['value'] <= log_merged['p5'])].copy()
low_logs['severity_low'] = log_merged.loc[low_logs.index, 'p5'] / np.maximum(low_logs['value'], 1e-12)
low_top = (
    low_logs
    .loc[:, ['timestamp', 'cmdb_id', 'log_name', 'value', 'p5', 'severity_low']]
    .sort_values('severity_low', ascending=False)
    .head(20)
    .reset_index(drop=True)
)

# 6) Aggregate table for the five services: anomaly_count (high+low), earliest_log_anomaly, top2_log_names_by_count
anoms = pd.concat([
    high_logs.loc[:, ['timestamp', 'cmdb_id', 'log_name']],
    low_logs.loc[:, ['timestamp', 'cmdb_id', 'log_name']]
], ignore_index=True, sort=False)

anom_counts = (
    anoms.groupby('cmdb_id', dropna=False)
    .agg(log_anomaly_count=('log_name', 'size'),
         earliest_log_anomaly=('timestamp', 'min'))
    .reset_index()
)

all_services_df = pd.DataFrame({'cmdb_id': services})
agg = all_services_df.merge(anom_counts, on='cmdb_id', how='left')
agg['log_anomaly_count'] = agg['log_anomaly_count'].fillna(0).astype(int)
agg['earliest_log_anomaly'] = agg['earliest_log_anomaly']  # NaT when no anomalies

# Compute top2 log_names by count per cmdb_id
logname_counts = anoms.groupby(['cmdb_id', 'log_name']).size().reset_index(name='count')
def top2_logs(cmdb):
    sub = logname_counts.loc[logname_counts['cmdb_id'] == cmdb]
    if sub.empty:
        return ""
    top = sub.sort_values('count', ascending=False).head(2)
    return ", ".join(f"{row.log_name}({int(row['count'])})" for _, row in top.iterrows())

agg['top2_log_names_by_count'] = agg['cmdb_id'].map(top2_logs)
agg_result = agg[['cmdb_id', 'log_anomaly_count', 'earliest_log_anomaly', 'top2_log_names_by_count']]

# 7) Check error_logs.csv for any rows whose cmdb_id is one of the five services within the incident window
if error_df.empty:
    error_matches = pd.DataFrame(columns=['timestamp', 'cmdb_id', 'message'])
    error_matches_empty_note = "error_logs.csv is empty or has no rows."
else:
    error_matches = error_df.loc[
        (error_df['timestamp'] >= start) &
        (error_df['timestamp'] <= end) &
        (error_df['cmdb_id'].isin(services)),
        ['timestamp', 'cmdb_id', 'message']
    ].copy().head(20).reset_index(drop=True)
    if error_matches.empty:
        error_matches_empty_note = "No error_log rows found for the five services in the incident window."
    else:
        error_matches_empty_note = None

# 8) First 10 rows (head) of the filtered log.csv for these five services
filtered_head = log_filtered.head(10).reset_index(drop=True)

# Return compact outputs
high_top, low_top, agg_result, error_matches if error_matches.shape[0]>0 else error_matches_empty_note, filtered_head

```
Out[5]:
```
Summary of log and error_log analysis for the five services (ts-auth-service, ts-order-service, ts-route-service, ts-train-service, ts-travel-service):

What was done
- Computed global P95 and P5 per (cmdb_id, log_name) over the full log series (no time filtering).
- Filtered log rows to the incident window 2024-01-24 08:18:50–08:48:50 UTC.
- Flagged high-side anomalies for log names containing "error" and low-side anomalies for log names containing "row_count".
- Checked error_logs.csv for matching entries in the window.

Key findings
- Error count metrics:
  - log.error_count values are zero during the window for the listed services, and the global p95 for these series is also 0.0. That yields no meaningful high-side error_count anomalies (severity is NaN because p95==0). In other words: no increased error_count observable in the normalized log metric.
- Row count drops (low-side anomalies):
  - Several services show row_count values well below their global P5 at 2024-01-24 08:21:00 UTC (examples):
    - ts-auth-service: row_count 52 vs p5 ≈ 262.8 (severity_low ≈ 5.05).
    - ts-route-service: row_count 181 vs p5 ≈ 831.8 (severity_low ≈ 4.60).
    - ts-travel-service: row_count 472 vs p5 ≈ 1176.2 (severity_low ≈ 2.49).
    - ts-order-service: row_count 114 vs p5 ≈ 256.0 (severity_low ≈ 2.25).
  - Some later row_count points are close to but still below P5 (e.g., around 08:36 and 08:45), but the largest drops were at ~08:21.
- Aggregate counts for the five services (within the window):
  - ts-auth-service, ts-order-service, ts-route-service, ts-travel-service each show 27 log anomalies (mostly log.error_count records and a couple of log.row_count low anomalies). ts-train-service shows 0 log anomalies in this window.
  - Earliest anomalies for the affected services are at ~2024-01-24 08:21:00 UTC.
- error_logs.csv:
  - No error_log rows were found for these five services in the incident window (the file is empty / no matches).

Interpretation / recommended next steps
- There is no evidence of elevated log.error_count in these services (the metric is zero), but there is a strong, simultaneous drop in log.row_count at ~08:21 for multiple services. That indicates either:
  - a drop in incoming requests / activity (fewer handled requests to log), or
  - a logging pipeline/sampling problem (logs not being emitted/collected), or
  - transient service-side short-circuiting causing fewer normal log lines.
- The lack of error_log entries (error_logs.csv empty) argues against many recorded textual errors in that file — but traces showed many error_rate anomalies earlier, so correlate results across traces and metrics.
- Next actions:
  - Correlate the row_count drop at ~08:21 with trace anomalies and metric anomalies (CPU, disk I/O) at the same times.
  - Inspect logging pipeline/agent for sampling or ingestion issues around 08:21.
  - Check service logs (raw logs) and traces for ts-auth, ts-order, ts-route, and ts-travel around 08:21 for root cause (requests failing earlier, or logging not happening).
  - Note ts-train-service shows no log anomalies in this window; either it was unaffected or missing log telemetry.

Quick preview (filtered log head)
- Example filtered rows at 08:21 show log.error_count = 0 and log.row_count values (e.g., ts-auth row_count=52, ts-order row_count=114, ts-route row_count=181, ts-travel row_count=472).

The original code execution output of IPython Kernel is also provided below for reference:

(                   timestamp            cmdb_id         log_name  value  p95  severity
0  2024-01-24 08:21:00+00:00    ts-auth-service  log.error_count      0  0.0       NaN
1  2024-01-24 08:21:00+00:00   ts-order-service  log.error_count      0  0.0       NaN
2  2024-01-24 08:21:00+00:00   ts-route-service  log.error_count      0  0.0       NaN
3  2024-01-24 08:21:00+00:00  ts-travel-service  log.error_count      0  0.0       NaN
4  2024-01-24 08:22:00+00:00    ts-auth-service  log.error_count      0  0.0       NaN
5  2024-01-24 08:22:00+00:00   ts-order-service  log.error_count      0  0.0       NaN
6  2024-01-24 08:22:00+00:00   ts-route-service  log.error_count      0  0.0       NaN
7  2024-01-24 08:22:00+00:00  ts-travel-service  log.error_count      0  0.0       NaN
8  2024-01-24 08:23:00+00:00    ts-auth-service  log.error_count      0  0.0       NaN
9  2024-01-24 08:23:00+00:00   ts-order-service  log.error_count      0  0.0       NaN
10 2024-01-24 08:23:00+00:00   ts-route-service  log.error_count      0  0.0       NaN
11 2024-01-24 08:23:00+00:00  ts-travel-service  log.error_count      0  0.0       NaN
12 2024-01-24 08:24:00+00:00    ts-auth-service  log.error_count      0  0.0       NaN
13 2024-01-24 08:24:00+00:00   ts-order-service  log.error_count      0  0.0       NaN
14 2024-01-24 08:24:00+00:00   ts-route-service  log.error_count      0  0.0       NaN
15 2024-01-24 08:24:00+00:00  ts-travel-service  log.error_count      0  0.0       NaN
16 2024-01-24 08:25:00+00:00    ts-auth-service  log.error_count      0  0.0       NaN
17 2024-01-24 08:25:00+00:00   ts-order-service  log.error_count      0  0.0       NaN
18 2024-01-24 08:25:00+00:00   ts-route-service  log.error_count      0  0.0       NaN
19 2024-01-24 08:25:00+00:00  ts-travel-service  log.error_count      0  0.0       NaN,                   timestamp            cmdb_id       log_name  value      p5  severity_low
0 2024-01-24 08:21:00+00:00    ts-auth-service  log.row_count     52   262.8      5.053846
1 2024-01-24 08:21:00+00:00   ts-route-service  log.row_count    181   831.8      4.595580
2 2024-01-24 08:21:00+00:00  ts-travel-service  log.row_count    472  1176.2      2.491949
3 2024-01-24 08:21:00+00:00   ts-order-service  log.row_count    114   256.0      2.245614
4 2024-01-24 08:36:00+00:00   ts-order-service  log.row_count    245   256.0      1.044898
5 2024-01-24 08:36:00+00:00  ts-travel-service  log.row_count   1127  1176.2      1.043656
6 2024-01-24 08:45:00+00:00    ts-auth-service  log.row_count    252   262.8      1.042857
7 2024-01-24 08:45:00+00:00   ts-route-service  log.row_count    811   831.8      1.025647,              cmdb_id  log_anomaly_count      earliest_log_anomaly                top2_log_names_by_count
0    ts-auth-service                 27 2024-01-24 08:21:00+00:00  log.error_count(25), log.row_count(2)
1   ts-order-service                 27 2024-01-24 08:21:00+00:00  log.error_count(25), log.row_count(2)
2   ts-route-service                 27 2024-01-24 08:21:00+00:00  log.error_count(25), log.row_count(2)
3   ts-train-service                  0                       NaT                                       
4  ts-travel-service                 27 2024-01-24 08:21:00+00:00  log.error_count(25), log.row_count(2), 'error_logs.csv is empty or has no rows.',                   timestamp            cmdb_id         log_name  value
0 2024-01-24 08:21:00+00:00    ts-auth-service  log.error_count      0
1 2024-01-24 08:21:00+00:00    ts-auth-service    log.row_count     52
2 2024-01-24 08:21:00+00:00   ts-order-service  log.error_count      0
3 2024-01-24 08:21:00+00:00   ts-order-service    log.row_count    114
4 2024-01-24 08:21:00+00:00   ts-route-service  log.error_count      0
5 2024-01-24 08:21:00+00:00   ts-route-service    log.row_count    181
6 2024-01-24 08:21:00+00:00  ts-travel-service  log.error_count      0
7 2024-01-24 08:21:00+00:00  ts-travel-service    log.row_count    472
8 2024-01-24 08:22:00+00:00    ts-auth-service  log.error_count      0
9 2024-01-24 08:22:00+00:00    ts-auth-service    log.row_count    340)```
```